# Fine-tune FinBERT for news direction

This notebook documents the small direction model used in the news pipeline. Given a company, a pillar and a headline, it predicts **negative**, **neutral** or **positive** impact for that company.

The default is review mode: it shows the verified result without downloading a model or starting training. Set `RUN_TRAINING = True` only when the checked private training CSV is available and a GPU runtime is selected.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

# This works when Jupyter starts in either the repository root or notebooks/.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "src").exists():
    raise FileNotFoundError("Start Jupyter in the repository root or notebooks/ folder.")

sys.path.insert(0, str(REPO_ROOT))
from src.finbert_training import (
    TrainingConfig,
    run_direction_experiment,
    validate_direction_dataset,
)

## Verified result

We selected the seed and epoch using development macro-F1. The combined 856-row test was evaluated only after that choice. The table below is a frozen record of the checked Colab T4 run, not a new run made while opening this notebook.

| Model | Accuracy | Macro-F1 |
|---|---:|---:|
| Majority baseline | 60.28% | 0.2507 |
| Generic FinBERT tone | 63.43% | 0.5762 |
| Fine-tuned FinBERT | **81.07%** | **0.7793** |

In [2]:
verified_models = pd.DataFrame([
    {"model": "Majority baseline", "accuracy": 0.602804, "macro_f1": 0.250729},
    {"model": "Generic FinBERT tone", "accuracy": 0.634346, "macro_f1": 0.576198},
    {"model": "Fine-tuned FinBERT", "accuracy": 0.810748, "macro_f1": 0.779289},
])

verified_seeds = pd.DataFrame([
    {"seed": 17, "best_epoch": 2, "dev_macro_f1": 0.729793},
    {"seed": 42, "best_epoch": 5, "dev_macro_f1": 0.705335},
    {"seed": 73, "best_epoch": 5, "dev_macro_f1": 0.661909},
])

display(verified_models.round(4))
display(verified_seeds.round(4))
print("Selected run: seed 17, epoch 2")

,model,accuracy,macro_f1
0,Majority baseline,0.6028,0.2507
1,Generic FinBERT tone,0.6343,0.5762
2,Fine-tuned FinBERT,0.8107,0.7793


,seed,best_epoch,dev_macro_f1
0,17,2,0.7298
1,42,5,0.7053
2,73,5,0.6619


Selected run: seed 17, epoch 2


## Data and split

The checked dataset contains 6,705 model-reviewed **silver-label** company–headline pairs: 5,730 train, 119 development and 856 test rows. Two independent model passes and a separate model adjudication resolved labels; these are not human gold labels. Related stories, exact duplicates and near-duplicate headline families stay in one split. The old locked test set was preserved, while new active-learning examples supplied a fresh development set and a 113-row challenge subset.

The model sees one short string:

```text
Company: {company}. Pillar: {pillar}. Headline: {headline}
```

To reduce class imbalance without ignoring neutral news, training uses the square root of inverse-frequency weight

$$
w_c = \sqrt{\frac{N}{K n_c}},
$$

where $N$ is the number of training rows, $K=3$ classes and $n_c$ is the number of training examples in class $c$.

In [3]:
RUN_TRAINING = False

config = TrainingConfig(
    base_model="ProsusAI/finbert",
    base_model_revision="4556d13015211d73dccd3fdd39d39232506f3e43",
    seeds=(17, 42, 73),
    max_epochs=5,
    patience=2,
    learning_rate=2e-5,
    train_batch_size=32,
    eval_batch_size=64,
    max_length=128,
)

# The labeled corpus is intentionally not distributed in this repository.
DATASET_PATH = REPO_ROOT / "data" / "private" / "finbert_direction_training.csv"
RUN_DIR = REPO_ROOT / "outputs" / "news_nlp" / "training_run"
EXPECTED_DATASET_SHA256 = "c0cf9f5c8cd1f34d77f630a14b14b5b4a291a2ef2d692905154312a1c2d3a0b1"

display(pd.Series({
    "run_training": RUN_TRAINING,
    "base_model": config.base_model,
    "seeds": list(config.seeds),
    "epochs_at_most": config.max_epochs,
    "early_stopping_patience": config.patience,
    "training_csv_present": DATASET_PATH.exists(),
}, name="value").to_frame())

,value
run_training,False
base_model,ProsusAI/finbert
seeds,"[17, 42, 73]"
epochs_at_most,5
early_stopping_patience,2
training_csv_present,False


## Optional reproduction

The helper module contains the reusable validation and training code, which keeps this notebook short. It checks the file hash, schema, labels, input template and leakage families before loading FinBERT. Each seed is trained independently; only development macro-F1 chooses the winner.

In [4]:
if RUN_TRAINING:
    dataset = validate_direction_dataset(
        DATASET_PATH,
        expected_sha256=EXPECTED_DATASET_SHA256,
        expected_split_rows=config.expected_split_rows,
    )
    display(pd.crosstab(dataset["split"], dataset["final_direction"]))

    result = run_direction_experiment(
        dataset,
        output_dir=RUN_DIR,
        config=config,
        dataset_sha256=EXPECTED_DATASET_SHA256,
    )
    display(pd.DataFrame(result["seed_development_runs"])[
        ["seed", "best_epoch", "dev_macro_f1"]
    ])
    display(pd.Series(result["combined_test"], name="held_out_test"))
else:
    print("Review mode: training was skipped and no model was downloaded.")

Review mode: training was skipped and no model was downloaded.


## What the result means

The selected model improved combined-test macro-F1 from **0.5762** for generic FinBERT tone to **0.7793**. On the harder 113-row active-learning challenge, macro-F1 was **0.6721**.

This is a headline-level direction signal, not a company ESG rating. Environmental evaluation was especially small (41 combined-test rows), so the pipeline keeps probabilities, confidence and coverage visible. The selected model archive is kept outside GitHub; its SHA-256 is `fc467b9ff6639aa4a77ae39ed7b486ecea36a6fc462206c91133308c525c6827`.